## Keegan Blakely Portfolio - Duplicate Record Matching
**Author:** Keegan Blakely  
**Date:** 2026-03-19  
**Description:** This notebook connects to a local MySQL database, pulls staged data,  
and scores potential duplicate records using various matching factors.


### Import Libraries

In [20]:
import mysql.connector
import pandas as pd
import re
from fuzzywuzzy import fuzz

### Parameters (match score weights & thresholds)
- Assign weights to apply to each category, weights should add up to 1
- Anyting below `possible_threshold` will be `Not Likely Match`

In [42]:
name_weight       = 0.15
birth_date_weight = 0.25
address_weight    = 0.2
phone_weight      = 0.3
email_weight      = 0.1

very_likely_threshold = 90
likely_threshold      = 80
possible_threshold    = 60

### Connect to Mysql Database

In [22]:
try:
    db_connection = mysql.connector.connect(
        host="localhost",  # MySQL server address
        user="root",       # MySQL username
        password="YOUR_PASSWORD_HERE",  # Replace with your password locally
        database="keegan_blakely_portfolio"  # Database name
    )

    if db_connection.is_connected():
        print("✅ Connected to the database successfully!")

except mysql.connector.Error as err:
    print(f"❌ Failed to connect to the database: {err}")

✅ Connected to the database successfully!


### Pull Data from Database

In [23]:
# Create a cursor with dictionary=True to get column names automatically
cursor = db_connection.cursor(dictionary=True)

# Execute query to select all data from the staging view
query = "SELECT * FROM stag_potential_duplicates;"
cursor.execute(query)

# Fetch all results and load into a Pandas DataFrame
df = pd.DataFrame(cursor.fetchall())

# Close the cursor now that we're done
cursor.close()

# Set Pandas to display all columns
pd.set_option('display.max_columns', None)

# Preview the first few rows
df.head()

,alt_id,alt_id_type,student_id,last_name,first_name,preferred_name,birth_date,gender_identity,international_status,last_name_std,first_name_std,preferred_name_std,gender_identity_std,address_id,address_lines,city,state,zip,country,address_lines_std,city_std,state_std,zip_std,country_std,email_1,email_2,email_1_std,email_2_std,phone_1,phone_2,phone_1_std,phone_2_std,enrolment_count,first_enrolment_date
0,1,ADM,48,Blakely,Keegan,None,1900-01-01,M,DO,blakely,keegan,None,m,157,Burquitlam Station,Coquitlam,BC,V3J 4K2,CA,burquitlam station,coquitlam,bc,v3j4k2,ca,blakelykeegan@gmail.com,kblakely@sfu.ca,blakelykeegan@gmail.com,kblakely@sfu.ca,778-988-1031,None,7789881031,None,8,2019-09-05
1,1,ADM,49,Blakely,Keegan K.,None,1900-01-01,M,DO,blakely,keegan k.,None,m,157,Burquitlam Station,Coquitlam,BC,V3J 4K2,CA,burquitlam station,coquitlam,bc,v3j4k2,ca,blakelykeegan@gmail.com,kblakely@jibc.ca,blakelykeegan@gmail.com,kblakely@jibc.ca,604.528.5692,(778) 988 1031,6045285692,7789881031,0,None
2,612345678,SIN,24,Carney,Mark J.,None,1965-03-16,M,IN,carney,mark j.,None,m,131,"Bank of England, Threadneedle Street",London,UK,EC2R 8AH,UK,bank of england threadneedle street,london,uk,ec2r8ah,uk,Mark.Carney@gmail.com,mark.carney@bankofengland.co.uk,mark.carney@gmail.com,mark.carney@bankofengland.co.uk,44-20-7601-5678,(44) 20 7601 1234,442076015678,442076011234,3,2025-01-10
3,612345678,SIN,25,Carney,Mark,None,1965-03-16,M,DO,carney,mark,None,m,145,24 Sussex Drive,Ottawa,ON,K1M1M4,CA,24 sussex drive,ottawa,on,k1m1m4,ca,mark.carney@gmail.com,mark.carney@canada.gc.ca,mark.carney@gmail.com,mark.carney@canada.gc.ca,1-613-555-0199,(44) 20 7601 1234,6135550199,442076011234,1,2020-09-01
4,737777645,PEN,50,Doe,Jane,None,1992-02-02,F,DO,doe,jane,None,f,158,28 King Street,Toronto,ON,M5H 3A1,CA,28 king street,toronto,on,m5h3a1,ca,JaniceSmith@gmail.com,None,janicesmith@gmail.com,None,1-604-604-6060,None,6046046060,None,0,None


### Generate Candidate Pairs
Group data by `alt_id` and create pairs of records to be scored for potential duplicates.

In [24]:
# Sort the DataFrame by alt_id to ensure duplicates are next to each other
df_sorted = df.sort_values(by='alt_id')

# Initialize a list to hold the pairs
pairs = []

# Group by alt_id and create pairs manually
for alt_id, group in df_sorted.groupby('alt_id'):
    if len(group) == 2:
        # Create a dictionary with standardized columns for the pair
        pair = {
            'alt_id': alt_id,
            'student_id_a': group.iloc[0]['student_id'],
            'student_id_b': group.iloc[1]['student_id'],
            'first_name_a': group.iloc[0]['first_name'],
            'first_name_b': group.iloc[1]['first_name'],
            'last_name_a': group.iloc[0]['last_name'],
            'last_name_b': group.iloc[1]['last_name'],
            'first_name_std_a': group.iloc[0]['first_name_std'],
            'first_name_std_b': group.iloc[1]['first_name_std'],
            'last_name_std_a': group.iloc[0]['last_name_std'],
            'last_name_std_b': group.iloc[1]['last_name_std'],
            'preferred_name_std_a': group.iloc[0]['preferred_name_std'],
            'preferred_name_std_b': group.iloc[1]['preferred_name_std'],
            'birth_date_a': group.iloc[0]['birth_date'],
            'birth_date_b': group.iloc[1]['birth_date'],
            'gender_identity_std_a': group.iloc[0]['gender_identity_std'],
            'gender_identity_std_b': group.iloc[1]['gender_identity_std'],
            'address_lines_std_a': group.iloc[0]['address_lines_std'],
            'address_lines_std_b': group.iloc[1]['address_lines_std'],
            'city_std_a': group.iloc[0]['city_std'],
            'city_std_b': group.iloc[1]['city_std'],
            'state_std_a': group.iloc[0]['state_std'],
            'state_std_b': group.iloc[1]['state_std'],
            'zip_std_a': group.iloc[0]['zip_std'],
            'zip_std_b': group.iloc[1]['zip_std'],
            'country_std_a': group.iloc[0]['country_std'],
            'country_std_b': group.iloc[1]['country_std'],
            'email_1_std_a': group.iloc[0]['email_1_std'],
            'email_1_std_b': group.iloc[1]['email_1_std'],
            'email_2_std_a': group.iloc[0]['email_2_std'],
            'email_2_std_b': group.iloc[1]['email_2_std'],
            'phone_1_std_a': group.iloc[0]['phone_1_std'],
            'phone_1_std_b': group.iloc[1]['phone_1_std'],
            'phone_2_std_a': group.iloc[0]['phone_2_std'],
            'phone_2_std_b': group.iloc[1]['phone_2_std'],
            'enrolment_count_a': group.iloc[0]['enrolment_count'],
            'enrolment_count_b': group.iloc[1]['enrolment_count'],
            'first_enrolment_date_a': group.iloc[0]['first_enrolment_date'],
            'first_enrolment_date_b': group.iloc[1]['first_enrolment_date'],
        }
        pairs.append(pair)

# Convert the list of pairs to a DataFrame
df_pairs = pd.DataFrame(pairs)

df_pairs.head()

,alt_id,student_id_a,student_id_b,first_name_a,first_name_b,last_name_a,last_name_b,first_name_std_a,first_name_std_b,last_name_std_a,last_name_std_b,preferred_name_std_a,preferred_name_std_b,birth_date_a,birth_date_b,gender_identity_std_a,gender_identity_std_b,address_lines_std_a,address_lines_std_b,city_std_a,city_std_b,state_std_a,state_std_b,zip_std_a,zip_std_b,country_std_a,country_std_b,email_1_std_a,email_1_std_b,email_2_std_a,email_2_std_b,phone_1_std_a,phone_1_std_b,phone_2_std_a,phone_2_std_b,enrolment_count_a,enrolment_count_b,first_enrolment_date_a,first_enrolment_date_b
0,1,48,49,Keegan,Keegan K.,Blakely,Blakely,keegan,keegan k.,blakely,blakely,None,None,1900-01-01,1900-01-01,m,m,burquitlam station,burquitlam station,coquitlam,coquitlam,bc,bc,v3j4k2,v3j4k2,ca,ca,blakelykeegan@gmail.com,blakelykeegan@gmail.com,kblakely@sfu.ca,kblakely@jibc.ca,7789881031,6045285692,None,7789881031,8,0,2019-09-05,None
1,111223333,43,44,Kal,Clark,El,Kent,kal,clark,el,kent,clark,None,1975-06-18,1975-06-18,m,m,344 clinton st,344 clinton st,metropolis,metropolis,ny,ny,10001,10001,us,us,kal.el@gmail.com,clark.kent@dailyplanet.com,None,None,2125557788,2125557788,None,2125553344,1,1,2020-09-01,2021-01-10
2,523456789,22,33,Daniel,Henrik,Sedin,Sedin,daniel,henrik,sedin,sedin,None,None,1980-09-26,1980-09-26,m,m,100 front st,123 maple st,vancouver,vancouver,bc,bc,v6b1a4,v7c2a6,ca,ca,danny22@nhl.com,hank.sedin@canucks.ca,None,None,6045553333,6045552222,None,None,3,3,2021-09-01,2021-09-01
3,612345678,24,25,Mark J.,Mark,Carney,Carney,mark j.,mark,carney,carney,None,None,1965-03-16,1965-03-16,m,m,bank of england threadneedle street,24 sussex drive,london,ottawa,uk,on,ec2r8ah,k1m1m4,uk,ca,mark.carney@gmail.com,mark.carney@gmail.com,mark.carney@bankofengland.co.uk,mark.carney@canada.gc.ca,442076015678,6135550199,442076011234,442076011234,3,1,2025-01-10,2020-09-01
4,737777645,50,51,Jane,Janice,Doe,Smith,jane,janice,doe,smith,None,jane,1992-02-02,1992-02-02,f,f,28 king street,28 king street,toronto,toronto,on,on,m5h3a1,m5h3a1,ca,ca,janicesmith@gmail.com,janedoe@gov.bc.ca,None,None,6046046060,6046046060,None,None,0,1,None,2021-06-20


### Last Name Match Score
Use fuzzy matching to compare last names of each pair.

In [25]:
# Apply fuzzy matching to last names
df_pairs['last_name_score'] = df_pairs.apply(
    lambda row: fuzz.ratio(row['last_name_std_a'], row['last_name_std_b']), axis=1
)

df_pairs.head()

,alt_id,student_id_a,student_id_b,first_name_a,first_name_b,last_name_a,last_name_b,first_name_std_a,first_name_std_b,last_name_std_a,last_name_std_b,preferred_name_std_a,preferred_name_std_b,birth_date_a,birth_date_b,gender_identity_std_a,gender_identity_std_b,address_lines_std_a,address_lines_std_b,city_std_a,city_std_b,state_std_a,state_std_b,zip_std_a,zip_std_b,country_std_a,country_std_b,email_1_std_a,email_1_std_b,email_2_std_a,email_2_std_b,phone_1_std_a,phone_1_std_b,phone_2_std_a,phone_2_std_b,enrolment_count_a,enrolment_count_b,first_enrolment_date_a,first_enrolment_date_b,last_name_score
0,1,48,49,Keegan,Keegan K.,Blakely,Blakely,keegan,keegan k.,blakely,blakely,None,None,1900-01-01,1900-01-01,m,m,burquitlam station,burquitlam station,coquitlam,coquitlam,bc,bc,v3j4k2,v3j4k2,ca,ca,blakelykeegan@gmail.com,blakelykeegan@gmail.com,kblakely@sfu.ca,kblakely@jibc.ca,7789881031,6045285692,None,7789881031,8,0,2019-09-05,None,100
1,111223333,43,44,Kal,Clark,El,Kent,kal,clark,el,kent,clark,None,1975-06-18,1975-06-18,m,m,344 clinton st,344 clinton st,metropolis,metropolis,ny,ny,10001,10001,us,us,kal.el@gmail.com,clark.kent@dailyplanet.com,None,None,2125557788,2125557788,None,2125553344,1,1,2020-09-01,2021-01-10,33
2,523456789,22,33,Daniel,Henrik,Sedin,Sedin,daniel,henrik,sedin,sedin,None,None,1980-09-26,1980-09-26,m,m,100 front st,123 maple st,vancouver,vancouver,bc,bc,v6b1a4,v7c2a6,ca,ca,danny22@nhl.com,hank.sedin@canucks.ca,None,None,6045553333,6045552222,None,None,3,3,2021-09-01,2021-09-01,100
3,612345678,24,25,Mark J.,Mark,Carney,Carney,mark j.,mark,carney,carney,None,None,1965-03-16,1965-03-16,m,m,bank of england threadneedle street,24 sussex drive,london,ottawa,uk,on,ec2r8ah,k1m1m4,uk,ca,mark.carney@gmail.com,mark.carney@gmail.com,mark.carney@bankofengland.co.uk,mark.carney@canada.gc.ca,442076015678,6135550199,442076011234,442076011234,3,1,2025-01-10,2020-09-01,100
4,737777645,50,51,Jane,Janice,Doe,Smith,jane,janice,doe,smith,None,jane,1992-02-02,1992-02-02,f,f,28 king street,28 king street,toronto,toronto,on,on,m5h3a1,m5h3a1,ca,ca,janicesmith@gmail.com,janedoe@gov.bc.ca,None,None,6046046060,6046046060,None,None,0,1,None,2021-06-20,0


### First Name Match Score (Including Preferred Name)
Compare first names and preferred names using fuzzy matching.

In [26]:
# Function to score first names, considering preferred names
def calculate_first_name_score(row):
    # Use empty string if any value is missing
    first_a = row['first_name_std_a'] or ''
    first_b = row['first_name_std_b'] or ''
    pref_a = row['preferred_name_std_a'] or ''
    pref_b = row['preferred_name_std_b'] or ''

    # Compare all combinations
    score_1 = fuzz.ratio(first_a, first_b)
    score_2 = fuzz.ratio(first_a, pref_b) if pref_b else 0
    score_3 = fuzz.ratio(pref_a, first_b) if pref_a else 0
    score_4 = fuzz.ratio(pref_a, pref_b) if pref_a and pref_b else 0

    # Perfect match shortcut
    if 100 in [score_1, score_2, score_3, score_4]:
        return 100
    return max(score_1, score_2, score_3, score_4)

# Apply function to create first_name_score column
df_pairs['first_name_score'] = df_pairs.apply(calculate_first_name_score, axis=1)

df_pairs.head()

,alt_id,student_id_a,student_id_b,first_name_a,first_name_b,last_name_a,last_name_b,first_name_std_a,first_name_std_b,last_name_std_a,last_name_std_b,preferred_name_std_a,preferred_name_std_b,birth_date_a,birth_date_b,gender_identity_std_a,gender_identity_std_b,address_lines_std_a,address_lines_std_b,city_std_a,city_std_b,state_std_a,state_std_b,zip_std_a,zip_std_b,country_std_a,country_std_b,email_1_std_a,email_1_std_b,email_2_std_a,email_2_std_b,phone_1_std_a,phone_1_std_b,phone_2_std_a,phone_2_std_b,enrolment_count_a,enrolment_count_b,first_enrolment_date_a,first_enrolment_date_b,last_name_score,first_name_score
0,1,48,49,Keegan,Keegan K.,Blakely,Blakely,keegan,keegan k.,blakely,blakely,None,None,1900-01-01,1900-01-01,m,m,burquitlam station,burquitlam station,coquitlam,coquitlam,bc,bc,v3j4k2,v3j4k2,ca,ca,blakelykeegan@gmail.com,blakelykeegan@gmail.com,kblakely@sfu.ca,kblakely@jibc.ca,7789881031,6045285692,None,7789881031,8,0,2019-09-05,None,100,80
1,111223333,43,44,Kal,Clark,El,Kent,kal,clark,el,kent,clark,None,1975-06-18,1975-06-18,m,m,344 clinton st,344 clinton st,metropolis,metropolis,ny,ny,10001,10001,us,us,kal.el@gmail.com,clark.kent@dailyplanet.com,None,None,2125557788,2125557788,None,2125553344,1,1,2020-09-01,2021-01-10,33,100
2,523456789,22,33,Daniel,Henrik,Sedin,Sedin,daniel,henrik,sedin,sedin,None,None,1980-09-26,1980-09-26,m,m,100 front st,123 maple st,vancouver,vancouver,bc,bc,v6b1a4,v7c2a6,ca,ca,danny22@nhl.com,hank.sedin@canucks.ca,None,None,6045553333,6045552222,None,None,3,3,2021-09-01,2021-09-01,100,33
3,612345678,24,25,Mark J.,Mark,Carney,Carney,mark j.,mark,carney,carney,None,None,1965-03-16,1965-03-16,m,m,bank of england threadneedle street,24 sussex drive,london,ottawa,uk,on,ec2r8ah,k1m1m4,uk,ca,mark.carney@gmail.com,mark.carney@gmail.com,mark.carney@bankofengland.co.uk,mark.carney@canada.gc.ca,442076015678,6135550199,442076011234,442076011234,3,1,2025-01-10,2020-09-01,100,73
4,737777645,50,51,Jane,Janice,Doe,Smith,jane,janice,doe,smith,None,jane,1992-02-02,1992-02-02,f,f,28 king street,28 king street,toronto,toronto,on,on,m5h3a1,m5h3a1,ca,ca,janicesmith@gmail.com,janedoe@gov.bc.ca,None,None,6046046060,6046046060,None,None,0,1,None,2021-06-20,0,100


### Gender Identity Mismatch Penalty Score
Apply a penalty if gender identities don’t match.

In [27]:
# Function to calculate gender score
def calculate_gender_score(row):
    # Compare gender identity for exact match or mismatch
    if row['gender_identity_std_a'] == row['gender_identity_std_b']:
        return 0  # Exact match, no penalty
    else:
        return -25  # Mismatch, strong penalty

# Apply the function to calculate gender score
df_pairs['gender_mismatch_penalty'] = df_pairs.apply(calculate_gender_score, axis=1)

df_pairs.head()

,alt_id,student_id_a,student_id_b,first_name_a,first_name_b,last_name_a,last_name_b,first_name_std_a,first_name_std_b,last_name_std_a,last_name_std_b,preferred_name_std_a,preferred_name_std_b,birth_date_a,birth_date_b,gender_identity_std_a,gender_identity_std_b,address_lines_std_a,address_lines_std_b,city_std_a,city_std_b,state_std_a,state_std_b,zip_std_a,zip_std_b,country_std_a,country_std_b,email_1_std_a,email_1_std_b,email_2_std_a,email_2_std_b,phone_1_std_a,phone_1_std_b,phone_2_std_a,phone_2_std_b,enrolment_count_a,enrolment_count_b,first_enrolment_date_a,first_enrolment_date_b,last_name_score,first_name_score,gender_mismatch_penalty
0,1,48,49,Keegan,Keegan K.,Blakely,Blakely,keegan,keegan k.,blakely,blakely,None,None,1900-01-01,1900-01-01,m,m,burquitlam station,burquitlam station,coquitlam,coquitlam,bc,bc,v3j4k2,v3j4k2,ca,ca,blakelykeegan@gmail.com,blakelykeegan@gmail.com,kblakely@sfu.ca,kblakely@jibc.ca,7789881031,6045285692,None,7789881031,8,0,2019-09-05,None,100,80,0
1,111223333,43,44,Kal,Clark,El,Kent,kal,clark,el,kent,clark,None,1975-06-18,1975-06-18,m,m,344 clinton st,344 clinton st,metropolis,metropolis,ny,ny,10001,10001,us,us,kal.el@gmail.com,clark.kent@dailyplanet.com,None,None,2125557788,2125557788,None,2125553344,1,1,2020-09-01,2021-01-10,33,100,0
2,523456789,22,33,Daniel,Henrik,Sedin,Sedin,daniel,henrik,sedin,sedin,None,None,1980-09-26,1980-09-26,m,m,100 front st,123 maple st,vancouver,vancouver,bc,bc,v6b1a4,v7c2a6,ca,ca,danny22@nhl.com,hank.sedin@canucks.ca,None,None,6045553333,6045552222,None,None,3,3,2021-09-01,2021-09-01,100,33,0
3,612345678,24,25,Mark J.,Mark,Carney,Carney,mark j.,mark,carney,carney,None,None,1965-03-16,1965-03-16,m,m,bank of england threadneedle street,24 sussex drive,london,ottawa,uk,on,ec2r8ah,k1m1m4,uk,ca,mark.carney@gmail.com,mark.carney@gmail.com,mark.carney@bankofengland.co.uk,mark.carney@canada.gc.ca,442076015678,6135550199,442076011234,442076011234,3,1,2025-01-10,2020-09-01,100,73,0
4,737777645,50,51,Jane,Janice,Doe,Smith,jane,janice,doe,smith,None,jane,1992-02-02,1992-02-02,f,f,28 king street,28 king street,toronto,toronto,on,on,m5h3a1,m5h3a1,ca,ca,janicesmith@gmail.com,janedoe@gov.bc.ca,None,None,6046046060,6046046060,None,None,0,1,None,2021-06-20,0,100,0


### Phone Number Match Score
Compare phone numbers across both records.

In [28]:
# Function to calculate phone number score
def calculate_phone_score(row):
    # Safely check if phone numbers are None
    phone_1_a = row['phone_1_std_a']
    phone_1_b = row['phone_1_std_b']
    phone_2_a = row['phone_2_std_a']
    phone_2_b = row['phone_2_std_b']
    
    # If both phone_2_std values are None, compare only phone_1_std values
    if phone_2_a is None and phone_2_b is None:
        if phone_1_a == phone_1_b:
            return 100  # Perfect match for phone_1_std
        else:
            return 0  # No match

    # Otherwise, compare all combinations of phone_1_std and phone_2_std values
    if (phone_1_a == phone_1_b or phone_1_a == phone_2_b or 
        phone_2_a == phone_1_b or phone_2_a == phone_2_b):
        return 100  # Perfect match
    else:
        return 0  # No match

# Apply the function to calculate phone score
df_pairs['phone_score'] = df_pairs.apply(calculate_phone_score, axis=1)

df_pairs.head()

,alt_id,student_id_a,student_id_b,first_name_a,first_name_b,last_name_a,last_name_b,first_name_std_a,first_name_std_b,last_name_std_a,last_name_std_b,preferred_name_std_a,preferred_name_std_b,birth_date_a,birth_date_b,gender_identity_std_a,gender_identity_std_b,address_lines_std_a,address_lines_std_b,city_std_a,city_std_b,state_std_a,state_std_b,zip_std_a,zip_std_b,country_std_a,country_std_b,email_1_std_a,email_1_std_b,email_2_std_a,email_2_std_b,phone_1_std_a,phone_1_std_b,phone_2_std_a,phone_2_std_b,enrolment_count_a,enrolment_count_b,first_enrolment_date_a,first_enrolment_date_b,last_name_score,first_name_score,gender_mismatch_penalty,phone_score
0,1,48,49,Keegan,Keegan K.,Blakely,Blakely,keegan,keegan k.,blakely,blakely,None,None,1900-01-01,1900-01-01,m,m,burquitlam station,burquitlam station,coquitlam,coquitlam,bc,bc,v3j4k2,v3j4k2,ca,ca,blakelykeegan@gmail.com,blakelykeegan@gmail.com,kblakely@sfu.ca,kblakely@jibc.ca,7789881031,6045285692,None,7789881031,8,0,2019-09-05,None,100,80,0,100
1,111223333,43,44,Kal,Clark,El,Kent,kal,clark,el,kent,clark,None,1975-06-18,1975-06-18,m,m,344 clinton st,344 clinton st,metropolis,metropolis,ny,ny,10001,10001,us,us,kal.el@gmail.com,clark.kent@dailyplanet.com,None,None,2125557788,2125557788,None,2125553344,1,1,2020-09-01,2021-01-10,33,100,0,100
2,523456789,22,33,Daniel,Henrik,Sedin,Sedin,daniel,henrik,sedin,sedin,None,None,1980-09-26,1980-09-26,m,m,100 front st,123 maple st,vancouver,vancouver,bc,bc,v6b1a4,v7c2a6,ca,ca,danny22@nhl.com,hank.sedin@canucks.ca,None,None,6045553333,6045552222,None,None,3,3,2021-09-01,2021-09-01,100,33,0,0
3,612345678,24,25,Mark J.,Mark,Carney,Carney,mark j.,mark,carney,carney,None,None,1965-03-16,1965-03-16,m,m,bank of england threadneedle street,24 sussex drive,london,ottawa,uk,on,ec2r8ah,k1m1m4,uk,ca,mark.carney@gmail.com,mark.carney@gmail.com,mark.carney@bankofengland.co.uk,mark.carney@canada.gc.ca,442076015678,6135550199,442076011234,442076011234,3,1,2025-01-10,2020-09-01,100,73,0,100
4,737777645,50,51,Jane,Janice,Doe,Smith,jane,janice,doe,smith,None,jane,1992-02-02,1992-02-02,f,f,28 king street,28 king street,toronto,toronto,on,on,m5h3a1,m5h3a1,ca,ca,janicesmith@gmail.com,janedoe@gov.bc.ca,None,None,6046046060,6046046060,None,None,0,1,None,2021-06-20,0,100,0,100


### Additional Email Cleaning and Email Match Score
Remove puctuation from first half of emails and score.

In [29]:
# Function to clean up emails (remove punctuation before the @ symbol)
def clean_email(email):
    if pd.isnull(email):
        return email  # Return NaN values as is
    # Split email into local and domain parts
    local, domain = email.split('@')
    # Remove punctuation from the local part (before @)
    local = re.sub(r'[^\w\s]', '', local)
    # Return cleaned email
    return f"{local}@{domain}"

# Apply the clean_email function to both email columns
df_pairs['email_1_std_a'] = df_pairs['email_1_std_a'].apply(clean_email)
df_pairs['email_1_std_b'] = df_pairs['email_1_std_b'].apply(clean_email)
df_pairs['email_2_std_a'] = df_pairs['email_2_std_a'].apply(clean_email)
df_pairs['email_2_std_b'] = df_pairs['email_2_std_b'].apply(clean_email)

# Function to calculate email score with None handling
def calculate_email_score(row):
    email_1_a = row['email_1_std_a']
    email_1_b = row['email_1_std_b']
    email_2_a = row['email_2_std_a']
    email_2_b = row['email_2_std_b']
    
    # Only compare non-None emails
    if email_2_a is None and email_2_b is None:
        # If both second emails are None, compare only email_1 values
        if email_1_a == email_1_b:
            return 100  # Perfect match for email_1
        else:
            return 0  # No match
    
    if email_2_a is None:
        # If only email_2_a is None, compare email_1_a with both email_1_b and email_2_b
        if email_1_a == email_1_b or email_1_a == email_2_b:
            return 100  # Match found
        else:
            return 0  # No match
    if email_2_b is None:
        # If only email_2_b is None, compare email_1_b with both email_1_a and email_2_a
        if email_1_a == email_1_b or email_2_a == email_1_b:
            return 100  # Match found
        else:
            return 0  # No match
    
    # If both email_2 values are not None, compare all combinations
    if (email_1_a == email_1_b or email_1_a == email_2_b or 
        email_2_a == email_1_b or email_2_a == email_2_b):
        return 100  # Perfect match
    else:
        return 0  # No match

# Apply the function to calculate email score
df_pairs['email_score'] = df_pairs.apply(calculate_email_score, axis=1)

df_pairs.head()

,alt_id,student_id_a,student_id_b,first_name_a,first_name_b,last_name_a,last_name_b,first_name_std_a,first_name_std_b,last_name_std_a,last_name_std_b,preferred_name_std_a,preferred_name_std_b,birth_date_a,birth_date_b,gender_identity_std_a,gender_identity_std_b,address_lines_std_a,address_lines_std_b,city_std_a,city_std_b,state_std_a,state_std_b,zip_std_a,zip_std_b,country_std_a,country_std_b,email_1_std_a,email_1_std_b,email_2_std_a,email_2_std_b,phone_1_std_a,phone_1_std_b,phone_2_std_a,phone_2_std_b,enrolment_count_a,enrolment_count_b,first_enrolment_date_a,first_enrolment_date_b,last_name_score,first_name_score,gender_mismatch_penalty,phone_score,email_score
0,1,48,49,Keegan,Keegan K.,Blakely,Blakely,keegan,keegan k.,blakely,blakely,None,None,1900-01-01,1900-01-01,m,m,burquitlam station,burquitlam station,coquitlam,coquitlam,bc,bc,v3j4k2,v3j4k2,ca,ca,blakelykeegan@gmail.com,blakelykeegan@gmail.com,kblakely@sfu.ca,kblakely@jibc.ca,7789881031,6045285692,None,7789881031,8,0,2019-09-05,None,100,80,0,100,100
1,111223333,43,44,Kal,Clark,El,Kent,kal,clark,el,kent,clark,None,1975-06-18,1975-06-18,m,m,344 clinton st,344 clinton st,metropolis,metropolis,ny,ny,10001,10001,us,us,kalel@gmail.com,clarkkent@dailyplanet.com,None,None,2125557788,2125557788,None,2125553344,1,1,2020-09-01,2021-01-10,33,100,0,100,0
2,523456789,22,33,Daniel,Henrik,Sedin,Sedin,daniel,henrik,sedin,sedin,None,None,1980-09-26,1980-09-26,m,m,100 front st,123 maple st,vancouver,vancouver,bc,bc,v6b1a4,v7c2a6,ca,ca,danny22@nhl.com,hanksedin@canucks.ca,None,None,6045553333,6045552222,None,None,3,3,2021-09-01,2021-09-01,100,33,0,0,0
3,612345678,24,25,Mark J.,Mark,Carney,Carney,mark j.,mark,carney,carney,None,None,1965-03-16,1965-03-16,m,m,bank of england threadneedle street,24 sussex drive,london,ottawa,uk,on,ec2r8ah,k1m1m4,uk,ca,markcarney@gmail.com,markcarney@gmail.com,markcarney@bankofengland.co.uk,markcarney@canada.gc.ca,442076015678,6135550199,442076011234,442076011234,3,1,2025-01-10,2020-09-01,100,73,0,100,100
4,737777645,50,51,Jane,Janice,Doe,Smith,jane,janice,doe,smith,None,jane,1992-02-02,1992-02-02,f,f,28 king street,28 king street,toronto,toronto,on,on,m5h3a1,m5h3a1,ca,ca,janicesmith@gmail.com,janedoe@gov.bc.ca,None,None,6046046060,6046046060,None,None,0,1,None,2021-06-20,0,100,0,100,0


### Additional Address Lines Cleaning and Address Match Score
Remove extra spaces and puctuation then use fuzzy matching to score.

In [30]:
# Function to clean address (just remove unnecessary spaces and punctuation)
def clean_address(address):
    if pd.isnull(address):
        return address  # Return NaN as is
    address = re.sub(r'[^\w\s,]', '', address)  # Remove punctuation but keep valid street characters (like commas)
    address = re.sub(r'\s+', ' ', address).strip()  # Clean up multiple spaces and trim
    return address

# Apply the clean_address function to the address columns
df_pairs['address_lines_std_a'] = df_pairs['address_lines_std_a'].apply(clean_address)
df_pairs['address_lines_std_b'] = df_pairs['address_lines_std_b'].apply(clean_address)

# Function to calculate address score using fuzzy matching
def calculate_address_score(row):
    address_a = row['address_lines_std_a']
    address_b = row['address_lines_std_b']
    
    # Use fuzzy matching to calculate similarity score
    return fuzz.ratio(address_a, address_b)

# Apply the function to calculate address score
df_pairs['address_score'] = df_pairs.apply(calculate_address_score, axis=1)

df_pairs.head()

,alt_id,student_id_a,student_id_b,first_name_a,first_name_b,last_name_a,last_name_b,first_name_std_a,first_name_std_b,last_name_std_a,last_name_std_b,preferred_name_std_a,preferred_name_std_b,birth_date_a,birth_date_b,gender_identity_std_a,gender_identity_std_b,address_lines_std_a,address_lines_std_b,city_std_a,city_std_b,state_std_a,state_std_b,zip_std_a,zip_std_b,country_std_a,country_std_b,email_1_std_a,email_1_std_b,email_2_std_a,email_2_std_b,phone_1_std_a,phone_1_std_b,phone_2_std_a,phone_2_std_b,enrolment_count_a,enrolment_count_b,first_enrolment_date_a,first_enrolment_date_b,last_name_score,first_name_score,gender_mismatch_penalty,phone_score,email_score,address_score
0,1,48,49,Keegan,Keegan K.,Blakely,Blakely,keegan,keegan k.,blakely,blakely,None,None,1900-01-01,1900-01-01,m,m,burquitlam station,burquitlam station,coquitlam,coquitlam,bc,bc,v3j4k2,v3j4k2,ca,ca,blakelykeegan@gmail.com,blakelykeegan@gmail.com,kblakely@sfu.ca,kblakely@jibc.ca,7789881031,6045285692,None,7789881031,8,0,2019-09-05,None,100,80,0,100,100,100
1,111223333,43,44,Kal,Clark,El,Kent,kal,clark,el,kent,clark,None,1975-06-18,1975-06-18,m,m,344 clinton st,344 clinton st,metropolis,metropolis,ny,ny,10001,10001,us,us,kalel@gmail.com,clarkkent@dailyplanet.com,None,None,2125557788,2125557788,None,2125553344,1,1,2020-09-01,2021-01-10,33,100,0,100,0,100
2,523456789,22,33,Daniel,Henrik,Sedin,Sedin,daniel,henrik,sedin,sedin,None,None,1980-09-26,1980-09-26,m,m,100 front st,123 maple st,vancouver,vancouver,bc,bc,v6b1a4,v7c2a6,ca,ca,danny22@nhl.com,hanksedin@canucks.ca,None,None,6045553333,6045552222,None,None,3,3,2021-09-01,2021-09-01,100,33,0,0,0,42
3,612345678,24,25,Mark J.,Mark,Carney,Carney,mark j.,mark,carney,carney,None,None,1965-03-16,1965-03-16,m,m,bank of england threadneedle street,24 sussex drive,london,ottawa,uk,on,ec2r8ah,k1m1m4,uk,ca,markcarney@gmail.com,markcarney@gmail.com,markcarney@bankofengland.co.uk,markcarney@canada.gc.ca,442076015678,6135550199,442076011234,442076011234,3,1,2025-01-10,2020-09-01,100,73,0,100,100,24
4,737777645,50,51,Jane,Janice,Doe,Smith,jane,janice,doe,smith,None,jane,1992-02-02,1992-02-02,f,f,28 king street,28 king street,toronto,toronto,on,on,m5h3a1,m5h3a1,ca,ca,janicesmith@gmail.com,janedoe@gov.bc.ca,None,None,6046046060,6046046060,None,None,0,1,None,2021-06-20,0,100,0,100,0,100


### City, State, Zip, and Country Match Scores

In [31]:
# Function to calculate exact match score for city, state, zip, and country
def calculate_exact_match_score(row, field_a, field_b):
    # Return 100 if they match, otherwise 0
    return 100 if row[field_a] == row[field_b] else 0

# Apply the function to calculate exact match scores for city, state, zip, and country
df_pairs['city_score'] = df_pairs.apply(calculate_exact_match_score, axis=1, field_a='city_std_a', field_b='city_std_b')
df_pairs['state_score'] = df_pairs.apply(calculate_exact_match_score, axis=1, field_a='state_std_a', field_b='state_std_b')
df_pairs['zip_score'] = df_pairs.apply(calculate_exact_match_score, axis=1, field_a='zip_std_a', field_b='zip_std_b')
df_pairs['country_score'] = df_pairs.apply(calculate_exact_match_score, axis=1, field_a='country_std_a', field_b='country_std_b')

df_pairs.head()

,alt_id,student_id_a,student_id_b,first_name_a,first_name_b,last_name_a,last_name_b,first_name_std_a,first_name_std_b,last_name_std_a,last_name_std_b,preferred_name_std_a,preferred_name_std_b,birth_date_a,birth_date_b,gender_identity_std_a,gender_identity_std_b,address_lines_std_a,address_lines_std_b,city_std_a,city_std_b,state_std_a,state_std_b,zip_std_a,zip_std_b,country_std_a,country_std_b,email_1_std_a,email_1_std_b,email_2_std_a,email_2_std_b,phone_1_std_a,phone_1_std_b,phone_2_std_a,phone_2_std_b,enrolment_count_a,enrolment_count_b,first_enrolment_date_a,first_enrolment_date_b,last_name_score,first_name_score,gender_mismatch_penalty,phone_score,email_score,address_score,city_score,state_score,zip_score,country_score
0,1,48,49,Keegan,Keegan K.,Blakely,Blakely,keegan,keegan k.,blakely,blakely,None,None,1900-01-01,1900-01-01,m,m,burquitlam station,burquitlam station,coquitlam,coquitlam,bc,bc,v3j4k2,v3j4k2,ca,ca,blakelykeegan@gmail.com,blakelykeegan@gmail.com,kblakely@sfu.ca,kblakely@jibc.ca,7789881031,6045285692,None,7789881031,8,0,2019-09-05,None,100,80,0,100,100,100,100,100,100,100
1,111223333,43,44,Kal,Clark,El,Kent,kal,clark,el,kent,clark,None,1975-06-18,1975-06-18,m,m,344 clinton st,344 clinton st,metropolis,metropolis,ny,ny,10001,10001,us,us,kalel@gmail.com,clarkkent@dailyplanet.com,None,None,2125557788,2125557788,None,2125553344,1,1,2020-09-01,2021-01-10,33,100,0,100,0,100,100,100,100,100
2,523456789,22,33,Daniel,Henrik,Sedin,Sedin,daniel,henrik,sedin,sedin,None,None,1980-09-26,1980-09-26,m,m,100 front st,123 maple st,vancouver,vancouver,bc,bc,v6b1a4,v7c2a6,ca,ca,danny22@nhl.com,hanksedin@canucks.ca,None,None,6045553333,6045552222,None,None,3,3,2021-09-01,2021-09-01,100,33,0,0,0,42,100,100,0,100
3,612345678,24,25,Mark J.,Mark,Carney,Carney,mark j.,mark,carney,carney,None,None,1965-03-16,1965-03-16,m,m,bank of england threadneedle street,24 sussex drive,london,ottawa,uk,on,ec2r8ah,k1m1m4,uk,ca,markcarney@gmail.com,markcarney@gmail.com,markcarney@bankofengland.co.uk,markcarney@canada.gc.ca,442076015678,6135550199,442076011234,442076011234,3,1,2025-01-10,2020-09-01,100,73,0,100,100,24,0,0,0,0
4,737777645,50,51,Jane,Janice,Doe,Smith,jane,janice,doe,smith,None,jane,1992-02-02,1992-02-02,f,f,28 king street,28 king street,toronto,toronto,on,on,m5h3a1,m5h3a1,ca,ca,janicesmith@gmail.com,janedoe@gov.bc.ca,None,None,6046046060,6046046060,None,None,0,1,None,2021-06-20,0,100,0,100,0,100,100,100,100,100


### Birth Date Match Score

In [32]:
# Function to calculate exact match score for birth_date
def calculate_birth_date_score(row):
    return 100 if row['birth_date_a'] == row['birth_date_b'] else 0

# Apply the function to calculate the birth date score for each pair
df_pairs['birth_date_score'] = df_pairs.apply(calculate_birth_date_score, axis=1)

df_pairs.head()

,alt_id,student_id_a,student_id_b,first_name_a,first_name_b,last_name_a,last_name_b,first_name_std_a,first_name_std_b,last_name_std_a,last_name_std_b,preferred_name_std_a,preferred_name_std_b,birth_date_a,birth_date_b,gender_identity_std_a,gender_identity_std_b,address_lines_std_a,address_lines_std_b,city_std_a,city_std_b,state_std_a,state_std_b,zip_std_a,zip_std_b,country_std_a,country_std_b,email_1_std_a,email_1_std_b,email_2_std_a,email_2_std_b,phone_1_std_a,phone_1_std_b,phone_2_std_a,phone_2_std_b,enrolment_count_a,enrolment_count_b,first_enrolment_date_a,first_enrolment_date_b,last_name_score,first_name_score,gender_mismatch_penalty,phone_score,email_score,address_score,city_score,state_score,zip_score,country_score,birth_date_score
0,1,48,49,Keegan,Keegan K.,Blakely,Blakely,keegan,keegan k.,blakely,blakely,None,None,1900-01-01,1900-01-01,m,m,burquitlam station,burquitlam station,coquitlam,coquitlam,bc,bc,v3j4k2,v3j4k2,ca,ca,blakelykeegan@gmail.com,blakelykeegan@gmail.com,kblakely@sfu.ca,kblakely@jibc.ca,7789881031,6045285692,None,7789881031,8,0,2019-09-05,None,100,80,0,100,100,100,100,100,100,100,100
1,111223333,43,44,Kal,Clark,El,Kent,kal,clark,el,kent,clark,None,1975-06-18,1975-06-18,m,m,344 clinton st,344 clinton st,metropolis,metropolis,ny,ny,10001,10001,us,us,kalel@gmail.com,clarkkent@dailyplanet.com,None,None,2125557788,2125557788,None,2125553344,1,1,2020-09-01,2021-01-10,33,100,0,100,0,100,100,100,100,100,100
2,523456789,22,33,Daniel,Henrik,Sedin,Sedin,daniel,henrik,sedin,sedin,None,None,1980-09-26,1980-09-26,m,m,100 front st,123 maple st,vancouver,vancouver,bc,bc,v6b1a4,v7c2a6,ca,ca,danny22@nhl.com,hanksedin@canucks.ca,None,None,6045553333,6045552222,None,None,3,3,2021-09-01,2021-09-01,100,33,0,0,0,42,100,100,0,100,100
3,612345678,24,25,Mark J.,Mark,Carney,Carney,mark j.,mark,carney,carney,None,None,1965-03-16,1965-03-16,m,m,bank of england threadneedle street,24 sussex drive,london,ottawa,uk,on,ec2r8ah,k1m1m4,uk,ca,markcarney@gmail.com,markcarney@gmail.com,markcarney@bankofengland.co.uk,markcarney@canada.gc.ca,442076015678,6135550199,442076011234,442076011234,3,1,2025-01-10,2020-09-01,100,73,0,100,100,24,0,0,0,0,100
4,737777645,50,51,Jane,Janice,Doe,Smith,jane,janice,doe,smith,None,jane,1992-02-02,1992-02-02,f,f,28 king street,28 king street,toronto,toronto,on,on,m5h3a1,m5h3a1,ca,ca,janicesmith@gmail.com,janedoe@gov.bc.ca,None,None,6046046060,6046046060,None,None,0,1,None,2021-06-20,0,100,0,100,0,100,100,100,100,100,100


### Create Composite Match Scores
Combine first and last name scores, and all location scores

In [33]:
# Composite name score: first and last
df_pairs['name_composite_score'] = 0.7 * df_pairs['first_name_score'] + 0.3 * df_pairs['last_name_score']

# Composite address score: zip, city, state, address line, country
df_pairs['address_composite_score'] = (
    0.4 * df_pairs['zip_score'] +
    0.3 * df_pairs['city_score'] +
    0.2 * df_pairs['address_score'] +
    0.05 * df_pairs['state_score'] +
    0.05 * df_pairs['country_score']
)

df_pairs.head()

,alt_id,student_id_a,student_id_b,first_name_a,first_name_b,last_name_a,last_name_b,first_name_std_a,first_name_std_b,last_name_std_a,last_name_std_b,preferred_name_std_a,preferred_name_std_b,birth_date_a,birth_date_b,gender_identity_std_a,gender_identity_std_b,address_lines_std_a,address_lines_std_b,city_std_a,city_std_b,state_std_a,state_std_b,zip_std_a,zip_std_b,country_std_a,country_std_b,email_1_std_a,email_1_std_b,email_2_std_a,email_2_std_b,phone_1_std_a,phone_1_std_b,phone_2_std_a,phone_2_std_b,enrolment_count_a,enrolment_count_b,first_enrolment_date_a,first_enrolment_date_b,last_name_score,first_name_score,gender_mismatch_penalty,phone_score,email_score,address_score,city_score,state_score,zip_score,country_score,birth_date_score,name_composite_score,address_composite_score
0,1,48,49,Keegan,Keegan K.,Blakely,Blakely,keegan,keegan k.,blakely,blakely,None,None,1900-01-01,1900-01-01,m,m,burquitlam station,burquitlam station,coquitlam,coquitlam,bc,bc,v3j4k2,v3j4k2,ca,ca,blakelykeegan@gmail.com,blakelykeegan@gmail.com,kblakely@sfu.ca,kblakely@jibc.ca,7789881031,6045285692,None,7789881031,8,0,2019-09-05,None,100,80,0,100,100,100,100,100,100,100,100,86.0,100.0
1,111223333,43,44,Kal,Clark,El,Kent,kal,clark,el,kent,clark,None,1975-06-18,1975-06-18,m,m,344 clinton st,344 clinton st,metropolis,metropolis,ny,ny,10001,10001,us,us,kalel@gmail.com,clarkkent@dailyplanet.com,None,None,2125557788,2125557788,None,2125553344,1,1,2020-09-01,2021-01-10,33,100,0,100,0,100,100,100,100,100,100,79.9,100.0
2,523456789,22,33,Daniel,Henrik,Sedin,Sedin,daniel,henrik,sedin,sedin,None,None,1980-09-26,1980-09-26,m,m,100 front st,123 maple st,vancouver,vancouver,bc,bc,v6b1a4,v7c2a6,ca,ca,danny22@nhl.com,hanksedin@canucks.ca,None,None,6045553333,6045552222,None,None,3,3,2021-09-01,2021-09-01,100,33,0,0,0,42,100,100,0,100,100,53.1,48.4
3,612345678,24,25,Mark J.,Mark,Carney,Carney,mark j.,mark,carney,carney,None,None,1965-03-16,1965-03-16,m,m,bank of england threadneedle street,24 sussex drive,london,ottawa,uk,on,ec2r8ah,k1m1m4,uk,ca,markcarney@gmail.com,markcarney@gmail.com,markcarney@bankofengland.co.uk,markcarney@canada.gc.ca,442076015678,6135550199,442076011234,442076011234,3,1,2025-01-10,2020-09-01,100,73,0,100,100,24,0,0,0,0,100,81.1,4.8
4,737777645,50,51,Jane,Janice,Doe,Smith,jane,janice,doe,smith,None,jane,1992-02-02,1992-02-02,f,f,28 king street,28 king street,toronto,toronto,on,on,m5h3a1,m5h3a1,ca,ca,janicesmith@gmail.com,janedoe@gov.bc.ca,None,None,6046046060,6046046060,None,None,0,1,None,2021-06-20,0,100,0,100,0,100,100,100,100,100,100,70.0,100.0


### Final Calculations

Calculate the weighted match score for each pair.  
If the pair meets the `possible_threshold`, determine the parent record:  
- The record with **more enrolments** is chosen as parent.  
- If enrolment counts are equal, the **earliest first enrolment date** is used as a tiebreaker.

In [43]:
# Calculate weighted match score with gender mismatch penalty
df_pairs['match_score'] = (
    df_pairs['name_composite_score'] * name_weight +
    df_pairs['address_composite_score'] * address_weight +
    df_pairs['birth_date_score'] * birth_date_weight +
    df_pairs['phone_score'] * phone_weight +
    df_pairs['email_score'] * email_weight +
    df_pairs['gender_mismatch_penalty']  # -25 if gender value does not match
)

# Create final output table with selected columns
final_output = df_pairs[[
    'alt_id',
    'student_id_a', 'student_id_b',
    'first_name_a', 'last_name_a',
    'first_name_b', 'last_name_b',
    'match_score',
    'enrolment_count_a', 'enrolment_count_b',
    'first_enrolment_date_a', 'first_enrolment_date_b'
]]

# Sort by match_score descending (highest likelihood first)
final_output = final_output.sort_values(by='match_score', ascending=False)

# Function to assign match likelihood category based on thresholds
def assign_match_likelihood(score):
    if score >= very_likely_threshold:
        return "Very Likely Match"
    elif score >= likely_threshold:
        return "Likely Match"
    elif score >= possible_threshold:
        return "Possible Match"
    else:
        return "Not Likely Match"

# Apply the function to create the new column
final_output['match_likelihood'] = final_output['match_score'].apply(assign_match_likelihood)

# Function to determine parent record
def determine_parent(row):
    if row['match_score'] < possible_threshold:
        return pd.NA, None  # No parent for low match scores
    
    # Compare enrolment counts
    if row['enrolment_count_a'] > row['enrolment_count_b']:
        return int(row['student_id_a']), "student_a"
    elif row['enrolment_count_b'] > row['enrolment_count_a']:
        return int(row['student_id_b']), "student_b"
    
    # Tie in enrolment counts → compare first enrolment dates
    date_a = row['first_enrolment_date_a']
    date_b = row['first_enrolment_date_b']
    
    if pd.isna(date_a) and not pd.isna(date_b):
        return int(row['student_id_b']), "student_b"
    elif pd.isna(date_b) and not pd.isna(date_a):
        return int(row['student_id_a']), "student_a"
    elif pd.isna(date_a) and pd.isna(date_b):
        return int(row['student_id_a']), "student_a"  # Default if both dates missing
    
    # Both dates exist → pick earliest
    if date_a <= date_b:
        return int(row['student_id_a']), "student_a"
    else:
        return int(row['student_id_b']), "student_b"

# Apply the parent determination function
final_output[['parent_student_id', 'parent_label']] = final_output.apply(determine_parent, axis=1, result_type='expand')

# Optional: drop helper columns to clean up final display
final_output = final_output[[
    'alt_id',
    'student_id_a', 'student_id_b',
    'first_name_a', 'last_name_a',
    'first_name_b', 'last_name_b',
    'match_score',
    'match_likelihood',
    'parent_student_id',
    'parent_label'
]]

## Final Output Table
Display final output table, OPTIONAL: export as csv

In [45]:
# Format match_score as two decimals and color match_likelihood
def style_match_table(df):
    df_display = df.copy()
    df_display['match_score'] = df_display['match_score'].round(2)

    # Color mapping based on match likelihood
    color_map = {
        "Very Likely Match": "#4CAF50",  # Green
        "Likely Match": "#FFEB3B",       # Yellow
        "Possible Match": "#FF9800",     # Orange
        "Not Likely Match": "#F44336"    # Red
    }

    styled = df_display.style \
        .applymap(lambda x: f'background-color: {color_map[x]}; color: black' 
                  if x in color_map else '', subset=['match_likelihood']) \
        .format({'match_score': "{:.2f}"}) \
        .set_properties(subset=['parent_label'], **{'font-weight': 'bold'}) \
        .set_caption("Duplicate Record Matching: Final Output")

    return styled

# Display the styled table
style_match_table(final_output)

# OPTIONAL: Export final_output DataFrame to CSV if needed
#final_output.to_csv("final_duplicate_matches_table.csv", index=False)

,alt_id,student_id_a,student_id_b,first_name_a,last_name_a,first_name_b,last_name_b,match_score,match_likelihood,parent_student_id,parent_label
0,1,48,49,Keegan,Blakely,Keegan K.,Blakely,97.90,Very Likely Match,48,student_a
5,987654321,41,42,Bruce,Wayne,Bruce,Wayne,97.04,Very Likely Match,41,student_a
1,111223333,43,44,Kal,El,Clark,Kent,86.98,Likely Match,43,student_a
4,737777645,50,51,Jane,Doe,Janice,Smith,85.50,Likely Match,51,student_b
3,612345678,24,25,Mark J.,Carney,Mark,Carney,78.12,Possible Match,24,student_a
2,523456789,22,33,Daniel,Sedin,Henrik,Sedin,42.64,Not Likely Match,,None


### Close SQL Connection
Run when you are done.

In [40]:
# Close the cursor if it exists
if 'cursor' in globals() and cursor is not None:
    cursor.close()
    print("✅ Cursor closed.")

# Close the database connection
if 'db_connection' in globals() and db_connection.is_connected():
    db_connection.close()
    print("✅ Database connection closed.")

✅ Cursor closed.
✅ Database connection closed.
